[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/reflections/icosahedron.ipynb)

# Three mirrors on the sphere

Three mirrors through the centre of a sphere, at angles of a half turn divided by 2, 3 and 5 to each other, cut out a triangle on it. Each pair of mirrors meets in an axis, and the product of their vectors is a rotor about it, whose powers return to a scalar after 2, 3 or 5 steps. Reflected in one another over and over, the mirrors cut the whole sphere into 120 copies of the triangle. The six copies around each threefold axis make one face of an icosahedron with its corners on the fivefold axes, and the mirror that runs from a fivefold axis to a twofold one draws its edges.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline

import numpy as np
from IPython.display import Image, display

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.animation import save_animation
from examples.math.reflections import render

ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
# The angles at which the mirrors meet: the first and the second, the first and the third, the second and
# the third.
FIRST_SECOND = np.pi / 2
FIRST_THIRD = np.pi / 3
SECOND_THIRD = np.pi / 5
RESOLUTION = 240
# Five rounds of reflections in the three mirrors fold any point of the sphere into the triangle.
ROUNDS = 5
FRAMES = 72
FRAME_MS = 60
TILT = 0.4

## 1. The mirrors

The mirror vectors point out of the triangle, and two of them pair to $-\cos(\pi / m)$ where their mirrors meet at the angle $\pi / m$. The first is $x$ and the second is $x$ turned in the $xy$ plane. The third has a part in the plane of the first two and a part along the plane's normal, its dual. In the plane, `second | plane.inverse()` pairs to one with the first vector and to zero with the second, and `plane.inverse() | first` the other way round: these reciprocal vectors, weighted by the pairings the third must have, are its part in the plane. Its length of one fixes its part along the normal.

In [ ]:
first = mv.x                                                                    # [] Vector
second = ((mv.x ^ mv.y) * (-(np.pi - FIRST_SECOND) / 2)).exp() >> first        # [] Vector
# The third mirror's part in the plane of the first two follows from its pairings with them, through the
# reciprocal vectors in that plane; what its unit length leaves goes along the plane's normal.
plane = first ^ second                                                          # [] Bivector
in_plane = -np.cos(FIRST_THIRD) * (second | plane.inverse()) - np.cos(SECOND_THIRD) * (plane.inverse() | first)   # [] Vector
normal = plane.normalized().dual()                                             # [] Vector
# Along the normal goes what the part in the plane leaves of a unit length.
height = (1 - in_plane.squared()).square_root()                                 # [] Scalar
third = in_plane + height * normal                                              # [] Vector
mirrors = stack([first, second, third])                                         # [mirrors] Vector

# The turns about the axes where two mirrors meet, named by how many of them make a scalar.
twofold = second * first                                                        # [] Rotor
threefold = third * first                                                       # [] Rotor
fivefold = third * second                                                       # [] Rotor

In [ ]:
print("pairings of the mirror vectors:")
print((mirrors[:, None] | mirrors[None, :]).to_array())

## 2. Folding the sphere into the triangle

A point lies on the wrong side of a mirror where it pairs negatively with the mirror's vector, and there it is reflected. Reflecting round after round in each of the three mirrors brings every point of the sphere into the triangle, and the number of reflections it took tells which copy of the triangle the point started in: an even number for the copies the turns reach, an odd number for their mirror images. Colouring by that parity alternates the 120 copies.

In [ ]:
def fold(points: Vector, mirrors: Vector, rounds: int) -> tuple[Vector, Scalar]:
    """The points reflected round after round in each mirror they lie on the wrong side of, and how
    many reflections each took."""
    flips = mv.scalar(np.zeros(points.shape + (1,)))                                 # [...] Scalar
    for _ in range(rounds):
        for index in range(mirrors.shape[0]):
            side = mirrors[index] | points                                      # [...] Scalar
            # One on the wrong side of the mirror, zero on the right side.
            wrong = (side.abs() - side) / (2 * side.abs())                      # [...] Scalar
            points = points + wrong * (-(mirrors[index] >> points) - points)
            flips = flips + wrong
    return points, flips


def turning(frames: int):
    """The sphere turning half around the axis where the first two mirrors meet, a symmetry of the
    tiling, seen tilted towards the viewer: per frame the folded points under each pixel and their
    reflection counts."""
    tilt = (mv.yz * (-TILT / 2)).exp()                                          # [] Rotor
    for angle in np.linspace(0.0, np.pi, frames, endpoint=False):
        spin = ((first ^ second) * (-angle / 2)).exp()                          # [] Rotor
        yield fold(spin >> (tilt >> screen), mirrors, ROUNDS)


# The visible half of the unit sphere, one point per pixel.
u, v = render.disk(RESOLUTION)
screen = mv.x * u + mv.y * v + mv.z * np.sqrt(np.clip(1 - u**2 - v**2, 0.0, None))   # [rows, columns] Vector

## 3. The icosahedron

The second mirror runs from a fivefold axis to a twofold one, opposite the threefold corner of the triangle; it is drawn bold, and outlines the faces of the icosahedron.

In [ ]:
display(Image(filename=save_animation(render.animate_sphere(screen, turning(FRAMES), mirrors, second), "icosahedron", FRAME_MS)))

In [ ]:
# checks
# The mirror vectors pair to -cos of the angle between their mirrors and have unit length; each turn's
# power of its order is a scalar of size one; every folded point lies on the right side of all three
# mirrors.
for one, other, angle in ((first, second, FIRST_SECOND), (first, third, FIRST_THIRD), (second, third, SECOND_THIRD)):
    np.testing.assert_allclose((one | other).to_array(), -np.cos(angle), atol=1e-12)
np.testing.assert_allclose((mirrors | mirrors).to_array(), 1.0, atol=1e-12)
for turn, order in ((twofold, 2), (threefold, 3), (fivefold, 5)):
    power = turn
    for _ in range(order - 1):
        power = power * turn                                                    # [] Rotor
    np.testing.assert_allclose(np.abs(power.select_grade(0).to_array()), 1.0, atol=1e-12)
    np.testing.assert_allclose(power.select_grade(2).kernel, 0.0, atol=1e-12)
folded, flips = next(turning(1))
inside = u**2 + v**2 < 1
assert ((mirrors | folded[..., None]).to_array()[inside] > -1e-12).all()